In [1]:
# ==========================================
# Task 4: Recommendation System
# Method: Collaborative Filtering (User-Based)
# ==========================================

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Step 1: Create a Synthetic Dataset
# -----------------------------------
# We create a dictionary where keys are users and values are their ratings for 5 movies.
# Ratings scale: 1 to 5.
# A rating of 0 indicates the user has NOT seen the movie yet.
data = {
    'User_A': [5, 4, 1, 0, 0],  # User A likes action/sci-fi, hasn't seen Notebook/La La Land
    'User_B': [4, 5, 0, 2, 0],  # User B is similar to User A
    'User_C': [1, 0, 5, 5, 4],  # User C likes romance/drama
    'User_D': [0, 2, 4, 5, 5],  # User D is similar to User C
    'User_E': [0, 0, 5, 4, 5]   # User E is also similar to C and D
}

# List of movie titles corresponding to the ratings above
movies = ['Matrix', 'Star Wars', 'Titanic', 'Notebook', 'La La Land']

# Create the DataFrame
# We transpose (.T) the data so that:
# - Rows represent USERS
# - Columns represent MOVIES
df = pd.DataFrame(data, index=movies).T

# Display the user-item matrix to verify structure
print("User-Movie Ratings Matrix:")
print(df)

User-Movie Ratings Matrix:
        Matrix  Star Wars  Titanic  Notebook  La La Land
User_A       5          4        1         0           0
User_B       4          5        0         2           0
User_C       1          0        5         5           4
User_D       0          2        4         5           5
User_E       0          0        5         4           5


In [2]:
# Step 2: Calculate Cosine Similarity
# -----------------------------------
# We use Cosine Similarity to measure how similar each user is to every other user.
# A score of 1.0 means identical ratings.
# A score near 0 means their tastes are completely different.

# Compute the similarity matrix based on the ratings in 'df'
user_similarity = cosine_similarity(df)

# Convert the result into a readable DataFrame
user_similarity_df = pd.DataFrame(user_similarity, index=df.index, columns=df.index)

print("\nUser Similarity Matrix (1.0 = Identical Tastes):")
print(user_similarity_df)


User Similarity Matrix (1.0 = Identical Tastes):
          User_A    User_B    User_C    User_D    User_E
User_A  1.000000  0.920087  0.188512  0.221313  0.094967
User_B  0.920087  1.000000  0.254967  0.356348  0.146795
User_C  0.188512  0.254967  1.000000  0.949133  0.977471
User_D  0.221313  0.356348  0.949133  1.000000  0.956296
User_E  0.094967  0.146795  0.977471  0.956296  1.000000


In [3]:
# Step 3: Define Recommendation Function
# --------------------------------------
def get_recommendations(user, df, similarity_matrix):
    """
    Generates movie recommendations for a specific user based on the most similar user.
    """

    # 1. Find the most similar user
    # We look at the similarity scores for the target 'user'
    similar_scores = similarity_matrix[user]

    # We drop the user themselves (score 1.0) and find the max remaining score
    most_similar_user = similar_scores.drop(user).idxmax()
    highest_score = similar_scores[most_similar_user]

    print(f"\n--- Recommendation Process for {user} ---")
    print(f"Most Similar User found: {most_similar_user} (Similarity Score: {highest_score:.2f})")

    # 2. Identify movies to recommend
    # Get ratings for both the target user and their 'twin' (most similar user)
    target_user_ratings = df.loc[user]
    similar_user_ratings = df.loc[most_similar_user]

    recommendations = []

    # Loop through all movies
    for movie in df.columns:
        # Condition 1: The target user has NOT seen it (Rating is 0)
        # Condition 2: The similar user DID like it (Rating > 3)
        if target_user_ratings[movie] == 0 and similar_user_ratings[movie] > 3:
            recommendations.append(movie)

    return recommendations

In [4]:
# Step 4: Generate Recommendations
# --------------------------------
# We will test the system for 'User_A'.
# Based on the data, User_A is most similar to User_B.
# User_B liked 'Star Wars' (which User_A already saw) and 'Matrix' (already saw).
# But wait—let's check 'User_C' or others to see clear recommendations.

# Let's try recommending for 'User_A'
recs_a = get_recommendations('User_A', df, user_similarity_df)
print(f"Recommended Movies: {recs_a}")

# Let's try recommending for 'User_D'
# User_D should be similar to User_C or User_E
recs_d = get_recommendations('User_D', df, user_similarity_df)
print(f"Recommended Movies: {recs_d}")


--- Recommendation Process for User_A ---
Most Similar User found: User_B (Similarity Score: 0.92)
Recommended Movies: []

--- Recommendation Process for User_D ---
Most Similar User found: User_E (Similarity Score: 0.96)
Recommended Movies: []
